In [1]:
import torch
from ultralytics import YOLO
import roboflow

print(f"GPU 체크: {torch.cuda.is_available()}") # True가 나와야 함
print(f"YOLOv8 버전: {torch.__version__}")

GPU 체크: True
YOLOv8 버전: 2.5.1+cu121


In [2]:
!nvidia-smi

Wed Jan 28 14:37:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.44                 Driver Version: 591.44         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   51C    P5             12W /   20W |       0MiB /   8188MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
import os
from roboflow import Roboflow
from ultralytics import YOLO

# ---------------------------------------------------------
# Step 1. 데이터셋 다운로드 
# ---------------------------------------------------------
rf = Roboflow(api_key="kHzDvVLoFaxNmCTjPHiX")
project = rf.workspace("lookie").project("my-first-project-sudpc")
version = project.version(3)
dataset = version.download("yolov8")

print(f"✅ 데이터셋 다운로드 완료 경로: {dataset.location}")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to My-First-Project-3 in yolov8:: 100%|██████████| 322/322 [00:00<00:00, 1544.85it/s]

✅ 데이터셋 다운로드 완료 경로: c:\Users\SSAFY\Desktop\S14P11B105\AI\My-First-Project-3


In [2]:
# ---------------------------------------------------------
# Step 2. 모델 학습 (Training)
# ---------------------------------------------------------
# CPU 추론용으로 가장 가볍고 빠른 'Nano' 모델(yolov8n.pt) 사용
model = YOLO('yolov8n.pt') 

print("🚀 학습을 시작합니다! (예상 소요 시간: 15~30분)")

results = model.train(
    data=f"{dataset.location}/data.yaml",  # 다운받은 데이터셋 경로 자동 연결
    epochs=100,                            # 학습 반복 횟수 (데이터가 적어서 금방 끝남)
    imgsz=640,                             # 아까 설정한 640 사이즈
    patience=20,                           # 20번 동안 성능 안 오르면 조기 종료 (시간 절약)
    batch=32,                              # 메모리 부족 에러 나면 8로 줄일 것
    device=0,                              # GPU 0번 사용 (필수)
    project='SFS_Logistics',               # 프로젝트 폴더명
    name='banana_detect_v1'                # 이번 실험 이름
)

🚀 학습을 시작합니다! (예상 소요 시간: 15~30분)
Ultralytics 8.4.8  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\SSAFY\Desktop\S14P11B105\AI\My-First-Project-3/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=banana_detect_v12, nbs=64,

In [6]:
# ---------------------------------------------------------
# Step 3. 모델 변환 (Export to ONNX)
# ---------------------------------------------------------
# AWS EC2(CPU)에서 빠르게 돌리기 위해 ONNX로 변환
print("📦 서비스 배포용 ONNX 파일로 변환 중...")
success = model.export(format='onnx', opset=12)

print("🎉 모든 작업 완료!")
print(f"생성된 파일 위치: SFS_Logistics/banana_detect_v1/weights/best.onnx")

📦 서비스 배포용 ONNX 파일로 변환 중...
Ultralytics 8.4.8  Python-3.10.19 torch-2.5.1+cu121 CPU (Intel Core(TM) Ultra 9 185H)
 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'C:\Users\SSAFY\Desktop\S14P11B105\AI\runs\detect\SFS_Logistics\banana_detect_v1\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (6.0 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---- ----------------------------------- 1.8/16.4 MB 10.1 MB/s eta 0:00:02
   ------------- -------------------------- 5.5/16.4 MB 14.0 MB/s eta 0:00:01
   ------------------------ --------------- 10.0/16.4 MB 16.4 MB/s eta 0:00:01
   -----------------